# <center> Advanced Python Exam 2026 </center>
# <center> variant 2 </center>

## Rules

1) The exam is open-book, thus you're allowed to have written notes, but you are still **not** allowed to carry and/or use flash drives and other\
electronic tech. and/or software of any kind to assist you

2) The exam will take 2 hours 40 minutes

3) You can ask any questions, but no answer is promised unless it is to a question related to the task description

4) Import the neccessary modules in the cell below the `Imports` label

5) You need to to ***ONE*** task in using psycopg2 and PostgreSQL and ***ONE*** task using only pandas

If you're going to be crying please do it silently so as not to disturb the rest of the students

# Imports

In [1]:
#your imports here

# Task 1 (50 pts). Students and Examinations

## Task description

Table: `Students`


| Column Name   | Type    |
|---------------|---------|
| student_id    | int     |
| student_name  | varchar |

student_id is the primary key (column with unique values) for this table.\
Each row of this table contains the ID and the name of one student in the school.
 

Table: `Subjects`


| Column Name  | Type    |
|--------------|---------|
| subject_name | varchar |

subject_name is the primary key (column with unique values) for this table.\
Each row of this table contains the name of one subject in the school.
 

Table: `Examinations`


| Column Name  | Type    |
|--------------|---------|
| student_id   | int     |
| subject_name | varchar |

There is no primary key (column with unique values) for this table. It may contain duplicates.\
Each student from the Students table takes every course from the Subjects table.\
Each row of this table indicates that a student with ID student_id attended the exam of subject_name.
 

Write a solution to find the number of times each student attended each exam.

Return the result table ordered by student_id and subject_name.

The result format is in the following example. Wrap your solution into the `sol_1` function

 

Example 1:

Input: 
Students table:

| student_id | student_name |
|------------|--------------|
| 1          | Alice        |
| 2          | Bob          |
| 13         | John         |
| 6          | Alex         |

Subjects table:

| subject_name |
|--------------|
| Math         |
| Physics      |
| Programming  |

Examinations table:

| student_id | subject_name |
|------------|--------------|
| 1          | Math         |
| 1          | Physics      |
| 1          | Programming  |
| 2          | Programming  |
| 1          | Physics      |
| 1          | Math         |
| 13         | Math         |
| 13         | Programming  |
| 13         | Physics      |
| 2          | Math         |
| 1          | Math         |

Output: 

| student_id | student_name | subject_name | attended_exams |
|------------|--------------|--------------|----------------|
| 1          | Alice        | Math         | 3              |
| 1          | Alice        | Physics      | 2              |
| 1          | Alice        | Programming  | 1              |
| 2          | Bob          | Math         | 1              |
| 2          | Bob          | Physics      | 0              |
| 2          | Bob          | Programming  | 1              |
| 6          | Alex         | Math         | 0              |
| 6          | Alex         | Physics      | 0              |
| 6          | Alex         | Programming  | 0              |
| 13         | John         | Math         | 1              |
| 13         | John         | Physics      | 1              |
| 13         | John         | Programming  | 1              |

Explanation: \
The result table should contain all students and all subjects.\
Alice attended the Math exam 3 times, the Physics exam 2 times, and the Programming exam 1 time.\
Bob attended the Math exam 1 time, the Programming exam 1 time, and did not attend the Physics exam.\
Alex did not attend any exams.\
John attended the Math exam 1 time, the Physics exam 1 time, and the Programming exam 1 time.

## Solution

In [4]:
def sol_1(students: pd.DataFrame, subjects: pd.DataFrame, examinations: pd.DataFrame) -> pd.DataFrame:
    """
    Returns for each student and each subject the number of times the student
    attended that exam.
    Ordered by student_id, subject_name.
    """
    # Cross join students and subjects
    cross = students.merge(subjects, how='cross')
    # Count attendance per student+subject
    attendance = (
        examinations.groupby(['student_id', 'subject_name'])
        .size()
        .reset_index(name='attended_exams')
    )
    # Left join to get counts (including 0 for no attendance)
    result = cross.merge(attendance, on=['student_id', 'subject_name'], how='left')
    result['attended_exams'] = result['attended_exams'].fillna(0).astype(int)
    # Order as required
    result = result.sort_values(['student_id', 'subject_name']).reset_index(drop=True)
    # Ensure columns order
    return result[['student_id', 'student_name', 'subject_name', 'attended_exams']]


### Test

Use the code block below to test your solutions

In [5]:
from pandas.testing import assert_frame_equal

print(f"Running test cases...\n")
passed = 0
failed = 0
for i in range(1, 101):
    status = f"\rTest {i}....."
    expected = pd.read_csv(fr'tests_task_1\exp_{i}.csv')
    st, sub, ex = pd.read_csv(fr'tests_task_1\st_{i}.csv'), pd.read_csv(fr'tests_task_1\sub_{i}.csv'), pd.read_csv(fr'tests_task_1\ex_{i}.csv')
    result = sol_1(st, sub, ex)
    try:
        assert_frame_equal(result.reset_index(drop=True), expected.reset_index(drop=True), check_dtype=False)
        print(status+"PASS", end='')
        passed += 1
    except AssertionError as e:
        print(status+"FAIL")
        failed += 1
        print("  Expected:")
        print(expected)
        print("  Got:")
        print(result)
        if len(expected) != len(result):
            print(f"  Lengths: expected {len(expected)}, got {len(result)}")
        break
else:
    print('\nAll tests succesfull!')
print(f"\nSummary: {passed} passed, {failed} failed out of 80 tests.")

Running test cases...

Test 100.....PASS
All tests succesfull!

Summary: 100 passed, 0 failed out of 80 tests.


# Task 2 (50 pts). Movie Rating

## Task description

Table: `Movies`

| Column Name   | Type    |
|---------------|---------|
| movie_id      | int     |
| title         | varchar |

movie_id is the primary key (column with unique values) for this table.\
title is the name of the movie.\
Each movie has a unique title.

Table: `Users`


| Column Name   | Type    |
|---------------|---------|
| user_id       | int     |
| name          | varchar |

user_id is the primary key (column with unique values) for this table.\
The column 'name' has unique values.

Table: `MovieRating`


| Column Name   | Type    |
|---------------|---------|
| movie_id      | int     |
| user_id       | int     |
| rating        | int     |
| created_at    | date    |

(movie_id, user_id) is the primary key (column with unique values) for this table.\
This table contains the rating of a movie by a user in their review.\
created_at is the user's review date. 
 

Write a solution to:

Find the name of the user who has rated the greatest number of movies. In case of a tie, return the lexicographically smaller user name.\
Find the movie name with the highest average rating in February 2020. In case of a tie, return the lexicographically smaller movie name.\
The result format is in the following example. Wrap the solution into the `sol_2` function

 

Example 1:

Input: 

Movies table:

| movie_id    |  title       |
|-------------|--------------|
| 1           | Avengers     |
| 2           | Frozen 2     |
| 3           | Joker        |

Users table:

| user_id     |  name        |
|-------------|--------------|
| 1           | Daniel       |
| 2           | Monica       |
| 3           | Maria        |
| 4           | James        |

MovieRating table:

| movie_id    | user_id      | rating       | created_at  |
|-------------|--------------|--------------|-------------|
| 1           | 1            | 3            | 2020-01-12  |
| 1           | 2            | 4            | 2020-02-11  |
| 1           | 3            | 2            | 2020-02-12  |
| 1           | 4            | 1            | 2020-01-01  |
| 2           | 1            | 5            | 2020-02-17  | 
| 2           | 2            | 2            | 2020-02-01  | 
| 2           | 3            | 2            | 2020-03-01  |
| 3           | 1            | 3            | 2020-02-22  | 
| 3           | 2            | 4            | 2020-02-25  | 

Output: 

| results      |
|--------------|
| Daniel       |
| Frozen 2     |

Explanation: \
Daniel and Monica have rated 3 movies ("Avengers", "Frozen 2" and "Joker") but Daniel is smaller lexicographically.\
Frozen 2 and Joker have a rating average of 3.5 in February but Frozen 2 is smaller lexicographically.

In [7]:
def sol_2(movies: pd.DataFrame, users: pd.DataFrame, movie_rating: pd.DataFrame) -> pd.DataFrame:

    # ---- Part 1: user with most ratings ----
    if not movie_rating.empty and not users.empty:
        user_counts = movie_rating.groupby('user_id').size().reset_index(name='cnt')
        user_counts = user_counts.merge(users, on='user_id', how='inner')

        if not user_counts.empty:
            max_cnt = user_counts['cnt'].max()
            top_user = (
                user_counts[user_counts['cnt'] == max_cnt]
                .sort_values('name')
                .iloc[0]['name']
            )
        else:
            top_user = None
    else:
        top_user = None

    # ---- Part 2: best movie in Feb 2020 ----
    feb = movie_rating[
        (movie_rating['created_at'] >= '2020-02-01') &
        (movie_rating['created_at'] <= '2020-02-29')
    ]

    if not feb.empty and not movies.empty:
        movie_avg = feb.groupby('movie_id')['rating'].mean().reset_index(name='avg')
        movie_avg = movie_avg.merge(movies, on='movie_id', how='inner')

        if not movie_avg.empty:
            max_avg = movie_avg['avg'].max()
            top_movie = (
                movie_avg[movie_avg['avg'] == max_avg]
                .sort_values('title')
                .iloc[0]['title']
            )
        else:
            top_movie = None
    else:
        top_movie = None

    return pd.DataFrame({'results': [top_user, top_movie]})

In [8]:
from pandas.testing import assert_frame_equal

print(f"Running test cases...\n")
passed = 0
failed = 0
for i in range(1, 94):
    status = f"\rTest {i}....."
    expected = pd.read_csv(fr'tests_task_2\exp_{i}.csv')
    movies, users, ratings = pd.read_csv(fr'tests_task_2\mv_{i}.csv'), pd.read_csv(fr'tests_task_2\usr_{i}.csv'), pd.read_csv(fr'tests_task_2\rt_{i}.csv')
    result = sol_2(movies, users, ratings)
    try:
        assert_frame_equal(result.reset_index(drop=True), expected.reset_index(drop=True), check_dtype=False)
        print(status+"PASS", end='')
        passed += 1
    except AssertionError as e:
        print(status+"FAIL")
        failed += 1
        print("  Expected:")
        print(expected)
        print("  Got:")
        print(result)
        if len(expected) != len(result):
            print(f"  Lengths: expected {len(expected)}, got {len(result)}")
        break
else:
    print('\nAll tests succesfull!')
    
print(f"\nSummary: {passed} passed, {failed} failed out of 93 tests.")

Running test cases...

Test 10.....PASS

C:\Users\Tecno\AppData\Local\Temp\ipykernel_3804\3989749973.py:12: FutureWarning: Mismatched null-like values None and nan found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  assert_frame_equal(result.reset_index(drop=True), expected.reset_index(drop=True), check_dtype=False)
C:\Users\Tecno\AppData\Local\Temp\ipykernel_3804\3989749973.py:12: FutureWarning: Mismatched null-like values None and nan found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  assert_frame_equal(result.reset_index(drop=True), expected.reset_index(drop=True), check_dtype=False)
C:\Users\Tecno\AppData\Local\Temp\ipykernel_3804\3989749973.py:12: FutureWarning: Mismatched null-like values None and nan found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  assert_frame_equal(result

Test 18.....PASS

C:\Users\Tecno\AppData\Local\Temp\ipykernel_3804\3989749973.py:12: FutureWarning: Mismatched null-like values None and nan found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  assert_frame_equal(result.reset_index(drop=True), expected.reset_index(drop=True), check_dtype=False)


Test 93.....PASS
All tests succesfull!

Summary: 93 passed, 0 failed out of 93 tests.
